In [2]:
import pandas as pd

DATA_PATH = "../../data/simplification_results3.csv"
SAVE_PATH = "../../data/cleaned.csv"

In [3]:
df = pd.read_csv(DATA_PATH)
cols = df.columns
prompt_names = []
metrics = []
for col in cols:
    split = col.split("__")
    if len(split) <= 1:
        continue
    prompt = split[0]
    metric = "__".join(split[1:])
    if prompt not in prompt_names:
        prompt_names += [prompt]
    if metric not in metrics:
        metrics += [metric]

prompt_names

['prompt_felix',
 'simple_summary',
 'literal_transcribe',
 'accessible_transcribe',
 'summarise']

If we stored the text better originally, It could now be turned into a pivot table.  
However from the current format thats absolutely a bit of a pain.  
Maybe redo it but with saving the prompt name as a row also.
Might need to do it by hand....

In [4]:
# I asked claude to help me write an unpivot function
def unpivot_prompts(df: pd.DataFrame, prompt_names, metrics) -> pd.DataFrame:
    prompt_variants = prompt_names

    chunks = []
    for variant in prompt_variants:
        cols = {'original_text': df['original_text']}
        cols['prompt'] = variant
        cols['transcribed_text'] = df[variant]

        for metric in metrics:
            source_col = f'{variant}__{metric}'
            # Normalise column name: drop double underscores
            target_col = metric.replace('__', '_')
            cols[target_col] = df[source_col]

        chunks.append(pd.DataFrame(cols))

    return pd.concat(chunks, ignore_index=True)

df = unpivot_prompts(df, prompt_names, metrics)
df.to_csv(SAVE_PATH)

In [ ]:
df = pd.read_csv(SAVE_PATH)

# I'm normalizing the fk_grade delta using 0-1 normalization, this is a choice, its not very well justified.
fk_max = df["fk_grade_delta"].max()
fk_min = df["fk_grade_delta"].min()

df["fk_normalized_score"] = (df["fk_grade_delta"] - fk_min)/(fk_max - fk_min)
# TODO: Above and below are quite hacky, and probably need to be done cleaner.
score_columns = []
for col in df.columns:
    if col.endswith("_score"):
        score_columns += [col]

# Define weights (default to 1.0 for each — adjust as needed)
weights = {col: 1.0 for col in score_columns}

# Compute weighted average
def add_weighted_average(df, score_columns, weights=None):
    if weights is None:
        weights = {col: 1.0 for col in score_columns}
    
    w = pd.Series(weights)
    df['average_score'] = df[score_columns].mul(w).sum(axis=1) / w.sum()
    return df

df = add_weighted_average(df, score_columns)

In [32]:
pre_aggregate = df[["prompt", "average_score"]]
aggregate = pre_aggregate.groupby("prompt").mean()
aggregate

,average_score
prompt,
accessible_transcribe,0.804244
literal_transcribe,0.840254
prompt_felix,0.827544
simple_summary,0.835317
summarise,0.763793


So my current findings say that literal_transcribe is the best on average score.

And summarise is the worst.

However I personally feel like the weights for readability and somesuch need to go up.

A lot of this does come down to personal preference and vibes...

In [36]:
# first i show all the current metrics so i can then change the weights
score_columns

['content_preserved_score',
 'actors_preserved_score',
 'no_meaning_change_score',
 'no_hallucination_score',
 'accessible_and_simple_score',
 'fk_normalized_score']

In [35]:
# Baseline weights for easy tweaking.
weights = {col: 1.0 for col in score_columns}

weights['fk_normalized_score'] = 5
weights['accessible_and_simple_score'] = 5

df = add_weighted_average(df, score_columns, weights)

pre_aggregate = df[["prompt", "average_score"]]
aggregate = pre_aggregate.groupby("prompt").mean()
aggregate

,average_score
prompt,
accessible_transcribe,0.866237
literal_transcribe,0.689434
prompt_felix,0.725690
simple_summary,0.774091
summarise,0.811302


I personally like these scores better, tough it is worth noting that this is all ran on an older version of the model, without the required context, so maybe the scores will work better in practice.

After this the next step would be verifying the results obtained, look at the same original text from a high and low scoring conversion, perhaps with a neato overvieuw of all the criteria. Tough ngl, that feels like it might add too much information.

Maybe just provide two randomly chosen texts. Fuck maybe i aggregated too soon allready, there could be some variance detected for certain prompts or metrics that might indicate warning signs.
This is all quite tricky, with the ammount of moving parts we have.

The parts to consider right now:
1. Different original texts.
2. Different conversion prompts.
3. Different metrics we measure on.
    * Each with a score and a value.
4. Different weights on each metric.






I think the variance might be a good indicator of something fishy about a certain prompt/metric.

Naively just high variance across a metric might indicate that it isn't capturing something well, tough tbf, these are just 0-1 judgements?

High variance amongst a prompts average scores might also indicate something funny going on, and should be investigated.

I guess the final workflow would be as follows:
- Introduce new metrics
- Check the metrics, to see if they measure what you want them to measure and behave rationally.
- Start itterating on prompt design, constantly checking if the scores still make sense.
- Potentially tweak the metrics, repeat.

So from this what is left to design:
- A checking system for metrics, those with high variance are probably better as metrics, but do check that they still work well. 
    - You might want to consider removing metrics with low variance, bc they arn't very informative.
    - So point out to the user: hi, this metric has low variance and can probably be removed, and take two samples, to let the user read the sample, and see the reasons for the scores, so they can check if the evaluation is any good.
- Then let the user A-B (or more) test different prompts and get their average scores.
    - high variance in average score can mean there's something funny going on with a prompt, so thats worth highlighting.
    - Otherwhise just verify the highest prompt's output, to see if you like it.
    - The user can then tweak the prompt, or the metrics, or both.

# Testing the metrics

In [ ]:

# === Metric Variance Check ===
# Metrics with higher variance are more informative — they discriminate between outputs.
# Low-variance metrics score almost everything the same way, giving little signal.
# Threshold is relative to max possible variance for a [0,1] score (0.25 for a coin flip).

LOW_VARIANCE_THRESHOLD = 0.04  # ~16% of max Bernoulli variance; adjust as needed
N_SAMPLES = 2

variances = df[score_columns].var().sort_values()

print("Metric variances (ascending):")
for metric, var in variances.items():
    flag = "  ← LOW" if var < LOW_VARIANCE_THRESHOLD else ""
    print(f"  {metric:<40} {var:.4f}{flag}")
print()

# For each low-variance metric, show samples so the user can judge metric quality
low_var_metrics = variances[variances < LOW_VARIANCE_THRESHOLD].index.tolist()

if not low_var_metrics:
    print("All metrics have sufficient variance — no candidates for removal.")
else:
    for metric in low_var_metrics:
        var = variances[metric]
        print("=" * 70)
        print(f"WARNING: '{metric}' has low variance ({var:.4f})")
        print(f"  → This metric gives nearly the same score to every output.")
        print(f"  → Consider: is it measuring something useful, or just always passing?")
        print(f"  → If it rarely changes, removing it won't affect rankings much.")
        print()

        # Look for a matching reason column (e.g. content_preserved_score → content_preserved_reason)
        reason_col = metric.replace("_score", "_reason")
        has_reason = reason_col in df.columns

        sample_cols = ["original_text", "transcribed_text", "prompt", metric]
        if has_reason:
            sample_cols.append(reason_col)

        sample = df[sample_cols].dropna(subset=[metric]).sample(
            min(N_SAMPLES, len(df)) 
        )

        for i, (_, row) in enumerate(sample.iterrows(), 1):
            print("-" * 70)
            print(f"  Sample {i}  (prompt: {row['prompt']})")
            print(f"    Original:   \n{str(row['original_text'])}")
            print(f"    Simplified: \n{str(row['transcribed_text'])}")
            print(f"\n    Score: {metric}: {row[metric]}")
            if has_reason:
                print(f"    Reason:     {str(row[reason_col])}")
            print()


Metric variances (ascending):
  accessible_and_simple_score              0.0000  ← LOW
  content_preserved_score                  0.0222  ← LOW
  fk_normalized_score                      0.0832
  no_hallucination_score                   0.1010
  actors_preserved_score                   0.1010
  no_meaning_change_score                  0.2343

  → This metric gives nearly the same score to every output.
  → Consider: is it measuring something useful, or just always passing?
  → If it rarely changes, removing it won't affect rankings much.

----------------------------------------------------------------------
  Sample 1  (prompt: summarise)
    Original:   
Torture and other cruel, inhuman or degrading treatment or punishment 29. Noting with concern reports of the systematic abuse of children by the police, the Committee recalls its general comment No. 13 (2011) and recommends that the State Party strengthen its efforts:
(a) To continue to implement the Juvenile Justice System Act, 2018

In [ ]:

# === Sanity-check sample: full metric breakdown for 2 random rows ===

reason_columns = [col for col in df.columns if col.endswith("_reason")]
sample = df[["original_text", "transcribed_text", "prompt"] + score_columns + reason_columns].sample(2)

for i, (_, row) in enumerate(sample.iterrows(), 1):
    print("=" * 70)
    print(f"Sample {i}  (prompt: {row['prompt']})")
    print(f"\n  Original:\n    {str(row['original_text'])}\n")
    print(f"  Simplified:\n    {str(row['transcribed_text'])}\n")
    print("  Metrics:")
    for metric in score_columns:
        reason_col = metric.replace("_score", "_reason")
        reason = str(row[reason_col]) if reason_col in df.columns else "—"
        print(f"    {metric:<40} {row[metric]}")
        print(f"      reason: {reason}")
    print()

Sample 1  (prompt: simple_summary)

  Original:
    Searches for disappeared persons and investigation and prosecution of cases of enforced disappearances 35. The Committee notes that the Office on Missing Persons is mandated to search for “missing and disappeared persons” and that it does not have the power to conduct criminal investigations, but must collaborate with the Criminal Investigation Department of the Police, which is responsible for investigating allegations of enforced disappearances, and the Attorney General’s Department, which is competent to prosecute these cases. It also notes that, according to the State Party, the Inspector General of Police has circulated instructions regarding the recording of complaints, and that disciplinary action is taken against police officers who do not comply. The Committee also notes that the High Court has exclusive jurisdiction to handle cases under the Enforced Disappearance Act and welcomes the State Party’s plan to establish an indep

# Testing the prompts.

In [42]:

# === Prompt A/B comparison ===
# Shows each prompt's mean and variance of average_score.
# High variance on a prompt means it works well on some texts and poorly on others —
# worth investigating before trusting its ranking.

HIGH_VARIANCE_THRESHOLD = 0.03  # adjust as needed

prompt_stats = df.groupby("prompt")["average_score"].agg(["mean", "var"]).sort_values("mean", ascending=False)
prompt_stats.columns = ["mean_score", "variance"]

print("Prompt comparison (ranked by mean score):")
print(f"  {'Prompt':<30} {'Mean':>6}  {'Var':>7}")
print(f"  {'-'*30} {'-'*6}  {'-'*7}")
for prompt, row in prompt_stats.iterrows():
    flag = "  ← HIGH VARIANCE" if row["variance"] > HIGH_VARIANCE_THRESHOLD else ""
    print(f"  {prompt:<30} {row['mean_score']:>6.3f}  {row['variance']:>7.4f}{flag}")
print()

high_var = prompt_stats[prompt_stats["variance"] > HIGH_VARIANCE_THRESHOLD]
if not high_var.empty:
    for prompt in high_var.index:
        print(f"WARNING: '{prompt}' has high variance ({high_var.loc[prompt, 'variance']:.4f})")
        print(f"  → Its score swings a lot across texts — check a few outputs before trusting its ranking.")
    print()


Prompt comparison (ranked by mean score):
  Prompt                           Mean      Var
  ------------------------------ ------  -------
  accessible_transcribe           0.866   0.0142
  summarise                       0.811   0.0059
  simple_summary                  0.774   0.0051
  prompt_felix                    0.726   0.0033
  literal_transcribe              0.689   0.0008



In [43]:

# === Verify best prompt ===
# Pull 2 random samples from the top-ranked prompt and show the full metric breakdown,
# so you can read the outputs and decide if you actually like what it's doing.

best_prompt = prompt_stats.index[0]
print(f"Best prompt: '{best_prompt}'  (mean={prompt_stats.loc[best_prompt, 'mean_score']:.3f})")
print(f"Read the outputs below and ask: does this look like good simplification?\n")

reason_columns = [col for col in df.columns if col.endswith("_reason")]
best_df = df[df["prompt"] == best_prompt]
sample = best_df[["original_text", "transcribed_text"] + score_columns + reason_columns].sample(2, random_state=7)

for i, (_, row) in enumerate(sample.iterrows(), 1):
    print("=" * 70)
    print(f"Sample {i}")
    print(f"\n  Original:\n    {str(row['original_text'])}\n")
    print(f"  Simplified:\n    {str(row['transcribed_text'])}\n")
    print("  Metrics:")
    for metric in score_columns:
        reason_col = metric.replace("_score", "_reason")
        reason = str(row[reason_col]) if reason_col in df.columns else "—"
        print(f"    {metric:<40} {row[metric]}")
        print(f"      reason: {reason}")
    print()


Best prompt: 'accessible_transcribe'  (mean=0.866)
Read the outputs below and ask: does this look like good simplification?

Sample 1

  Original:
    Abuse, neglect, sexual abuse and exploitation 26. In the light of its general comment No. 13 (2011), the Committee recommends that the State Party:
(a) Finalize the adoption of a strategy to address and prevent all forms of violence against children throughout its territory and to strengthen the child protection infrastructure at the local level to respond to all forms of violence against children;
(b) Ensure that a child-friendly system is in place for the mandatory reporting of all forms of violence against children, including by implementing it effectively throughout the country, identifying accessible focal points that should receive such reports and promoting awareness among parents, caregivers, professionals and children themselves on the importance of reporting and early intervention in cases of abuse and violence against children